# GUI Notebook:

## PyQT Example:

In [ ]:
# line_plotter.py
# Requires: PyQt5, matplotlib, numpy
# Run: python line_plotter.py

import sys
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout,
    QLineEdit, QLabel, QPushButton, QMessageBox
)
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class MplCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(5, 4), dpi=100)
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)


class LinePlotter(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Slope Plotter (y = m x + b)")

        # --- Inputs ---
        self.slope_input = QLineEdit()
        self.slope_input.setPlaceholderText("m (slope), e.g. 2.5")
        self.intercept_input = QLineEdit()
        self.intercept_input.setPlaceholderText("b (intercept, optional; default 0)")

        inputs = QHBoxLayout()
        inputs.addWidget(QLabel("m:"))
        inputs.addWidget(self.slope_input)
        inputs.addWidget(QLabel("b:"))
        inputs.addWidget(self.intercept_input)

        # --- Buttons ---
        self.plot_btn = QPushButton("Plot")
        self.clear_btn = QPushButton("Clear")
        btns = QHBoxLayout()
        btns.addWidget(self.plot_btn)
        btns.addWidget(self.clear_btn)

        # --- Canvas ---
        self.canvas = MplCanvas()

        # --- Layout ---
        layout = QVBoxLayout()
        layout.addLayout(inputs)
        layout.addLayout(btns)
        layout.addWidget(self.canvas)
        self.setLayout(layout)

        # --- Signals ---
        self.plot_btn.clicked.connect(self.plot_line)
        self.clear_btn.clicked.connect(self.clear_plot)

        # Initial hint plot
        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.draw()

    def plot_line(self):
        try:
            m = float(self.slope_input.text())
        except ValueError:
            self._err("Please enter a valid number for slope (m).")
            return

        b_text = self.intercept_input.text().strip()
        b = float(b_text) if b_text not in ("", None) else 0.0

        x = np.linspace(-10, 10, 400)
        y = m * x + b

        self.canvas.ax.clear()
        self.canvas.ax.plot(x, y, linewidth=2)
        self.canvas.ax.set_xlabel("x")
        self.canvas.ax.set_ylabel("y")
        self.canvas.ax.set_title(f"y = {m}x + {b}")
        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.ax.axhline(0, linewidth=1)
        self.canvas.ax.axvline(0, linewidth=1)
        self.canvas.fig.tight_layout()
        self.canvas.draw()

    def clear_plot(self):
        self.canvas.ax.clear()
        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.draw()

    def _err(self, msg):
        QMessageBox.critical(self, "Input Error", msg)


def main():
    app = QApplication(sys.argv)
    w = LinePlotter()
    w.resize(700, 500)
    w.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


In [ ]:
# line_plotter_v2.py
# Requires: PyQt5, matplotlib, numpy

import sys
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout,
    QLineEdit, QLabel, QPushButton, QMessageBox, QTextEdit
)
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure


class MplCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(5, 4), dpi=100)
        self.ax = self.fig.add_subplot(111)
        super().__init__(self.fig)


class LinePlotter(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Line Plotter (y = m x + b)")

        # --- Inputs ---
        self.slope_input = QLineEdit()
        self.slope_input.setPlaceholderText("m (slope)")
        self.intercept_input = QLineEdit()
        self.intercept_input.setPlaceholderText("b (intercept)")

        self.xmin_input = QLineEdit()
        self.xmin_input.setPlaceholderText("x min")
        self.xmax_input = QLineEdit()
        self.xmax_input.setPlaceholderText("x max")
        self.ymin_input = QLineEdit()
        self.ymin_input.setPlaceholderText("y min")
        self.ymax_input = QLineEdit()
        self.ymax_input.setPlaceholderText("y max")

        top_inputs = QHBoxLayout()
        for lbl, wid in zip(
            ["m:", "b:", "x min:", "x max:", "y min:", "y max:"],
            [self.slope_input, self.intercept_input,
             self.xmin_input, self.xmax_input, self.ymin_input, self.ymax_input]
        ):
            top_inputs.addWidget(QLabel(lbl))
            top_inputs.addWidget(wid)

        # --- Buttons ---
        self.plot_btn = QPushButton("Plot")
        self.clear_btn = QPushButton("Clear")
        btns = QHBoxLayout()
        btns.addWidget(self.plot_btn)
        btns.addWidget(self.clear_btn)

        # --- Output ---
        self.output = QTextEdit()
        self.output.setReadOnly(True)
        self.output.setPlaceholderText("Output will appear here...")

        # --- Canvas ---
        self.canvas = MplCanvas()

        # --- Layout ---
        layout = QVBoxLayout()
        layout.addLayout(top_inputs)
        layout.addLayout(btns)
        layout.addWidget(self.canvas)
        layout.addWidget(QLabel("Output:"))
        layout.addWidget(self.output)
        self.setLayout(layout)

        # --- Connections ---
        self.plot_btn.clicked.connect(self.plot_line)
        self.clear_btn.clicked.connect(self.clear_plot)

        # Initial setup
        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.draw()

    def plot_line(self):
        try:
            m = float(self.slope_input.text())
        except ValueError:
            self._err("Please enter a valid number for slope (m).")
            return

        b_text = self.intercept_input.text().strip()
        b = float(b_text) if b_text else 0.0

        # Graph bounds
        try:
            xmin = float(self.xmin_input.text()) if self.xmin_input.text() else -10
            xmax = float(self.xmax_input.text()) if self.xmax_input.text() else 10
            ymin = float(self.ymin_input.text()) if self.ymin_input.text() else None
            ymax = float(self.ymax_input.text()) if self.ymax_input.text() else None
        except ValueError:
            self._err("Bounds must be numbers.")
            return

        x = np.linspace(xmin, xmax, 400)
        y = m * x + b

        # Compute x-intercept
        if m != 0:
            x_int = -b / m
            y_int = 0
            intercept_str = f"X-intercept: x = {x_int:.2f}"
        else:
            intercept_str = "No x-intercept (horizontal line)"

        # --- Plot ---
        self.canvas.ax.clear()
        self.canvas.ax.plot(x, y, 'b', linewidth=2)
        self.canvas.ax.axhline(0, color='k', lw=1)
        self.canvas.ax.axvline(0, color='k', lw=1)
        self.canvas.ax.set_xlabel("x")
        self.canvas.ax.set_ylabel("y")
        self.canvas.ax.set_title(f"y = {m}x + {b}")

        if ymin is not None and ymax is not None:
            self.canvas.ax.set_ylim(ymin, ymax)

        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.fig.tight_layout()
        self.canvas.draw()

        # --- Output text ---
        self.output.setPlainText(
            f"Equation: y = {m}x + {b}\n"
            f"{intercept_str}\n"
            f"Displayed range: x ∈ [{xmin}, {xmax}]"
        )

    def clear_plot(self):
        self.canvas.ax.clear()
        self.canvas.ax.grid(True, alpha=0.3)
        self.canvas.draw()
        self.output.clear()

    def _err(self, msg):
        QMessageBox.critical(self, "Input Error", msg)


def main():
    app = QApplication(sys.argv)
    w = LinePlotter()
    w.resize(900, 600)
    w.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


In [1]:
# magnet_gui.py
# Requires: PyQt5, matplotlib, numpy
# Run: python magnet_gui.py

import sys, ast
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, QFormLayout,
    QLineEdit, QTextEdit, QLabel, QPushButton, QCheckBox, QMessageBox,
    QComboBox
)
from PyQt5.QtCore import Qt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec


# ==================== Math / Physics helpers (from your script) ====================

def plotter(lower, upper, rad, point_num):
    phi_list = np.linspace(np.deg2rad(lower), np.deg2rad(upper), point_num)
    x = rad*np.cos(phi_list)
    y = rad*np.sin(phi_list)
    return x, y

def circ_plot(ax, rad, color='k--'):
    circ = np.linspace(0, 2*np.pi, 500)
    x = rad*np.cos(circ)
    y = rad*np.sin(circ)
    ax.plot(x, y, color, zorder=1)

def B_harm(x, y, x_a, y_a, I, mu0, n_max):
    a = np.hypot(x_a, y_a)
    r = np.hypot(x, y)
    phi = np.arctan2(y_a, x_a)
    theta = np.arctan2(y, x)
    ang = phi - theta
    r = np.where(r == 0, 1e-10, r)

    B_r_in = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(r / a)**(n - 1) * np.sin(n * ang) for n in range(1, n_max+1)], axis=0)
    B_theta_in = (-1)*((mu0 * I) / (2 * np.pi * a)) * np.sum([(r / a)**(n - 1) * np.cos(n * ang) for n in range(1, n_max+1)], axis=0)

    B_r_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(a / r)**(n + 1) * np.sin(n * ang) for n in range(0, n_max+1)], axis=0)
    B_theta_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(a / r)**(n + 1) * np.cos(n * ang) for n in range(0, n_max+1)], axis=0)

    B_r = np.where(r > a, B_r_out, B_r_in)
    B_theta = np.where(r > a, B_theta_out, B_theta_in)

    B_x = B_r * np.cos(theta) - B_theta * np.sin(theta)
    B_y = B_r * np.sin(theta) + B_theta * np.cos(theta)
    return B_x, B_y

def Harmonic(R0, a, phi, n, I_wire, mu0):
    # Returns Gauss contribution (~1e4 factor included)
    return (mu0*I_wire/2*np.pi) * (10**4) * ((R0 / a)**n) * np.cos((n + 1) * phi)


# ==================== Matplotlib Canvas ====================

class MplCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(7, 6), dpi=100)
        # Reserve a fixed colorbar axis using GridSpec
        gs = GridSpec(1, 2, width_ratios=[20, 1], figure=self.fig)
        self.ax = self.fig.add_subplot(gs[0, 0])
        self.cax = self.fig.add_subplot(gs[0, 1])
        # Start with an empty colorbar axis
        self.cax.set_visible(True)
        self.cax.set_frame_on(False)
        self.cax.tick_params(left=False, labelleft=False)
        super().__init__(self.fig)


# ==================== Main GUI ====================

class MagnetGUI(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Magnet Field GUI (Layers as Input)")

        # ----- Inputs -----
        self.layers_edit = QTextEdit()
        self.layers_edit.setPlaceholderText("Paste your layers array here...\n"
                                            "Example (sextupole):\n"
                                            "[\n"
                                            "  [[3,6], [(3,13), (50,70)]],\n"
                                            "  [[3,6], [(3,13), (50,70)]],\n"
                                            "  [[3,6], [(3,13), (50,70)]]\n"
                                            "]")
        # Default example (sextupole)
        self.layers_edit.setPlainText(
            "[[ [3,6], [(3,13), (50,70)] ],\n"
            " [ [3,6], [(3,13), (50,70)] ],\n"
            " [ [3,6], [(3,13), (50,70)] ]]\n"
        )

        form = QFormLayout()
        self.nmax = QLineEdit("10")
        self.aper_rad = QLineEdit("2")
        self.dr = QLineEdit("0.1")
        self.t = QLineEdit("0.07")
        self.ref_rad = QLineEdit("1")
        self.I0 = QLineEdit("500")

        self.mag_type = QComboBox()
        self.mag_type.addItems(["Dipole (1)", "Quadrupole (2)", "Sextupole (3)"])
        self.mag_type.setCurrentIndex(2)  # Sextupole default

        self.surf_plot = QCheckBox("Surface plot (field magnitude)")
        self.surf_plot.setChecked(False)
        self.all_quads = QCheckBox("Show all quadrants")
        self.all_quads.setChecked(False)

        # Optional manual bounds
        self.xmin = QLineEdit("")
        self.xmax = QLineEdit("")
        self.ymin = QLineEdit("")
        self.ymax = QLineEdit("")

        form.addRow("n_max:", self.nmax)
        form.addRow("Aperture radius (aper_rad):", self.aper_rad)
        form.addRow("Radial spacing (dr):", self.dr)
        form.addRow("Layer thickness (t):", self.t)
        form.addRow("Reference radius (ref_rad):", self.ref_rad)
        form.addRow("Current magnitude (I0):", self.I0)
        form.addRow("Magnet type:", self.mag_type)
        form.addRow(self.surf_plot)
        form.addRow(self.all_quads)
        form.addRow(QLabel("Manual bounds (optional):"))
        hb1 = QHBoxLayout()
        hb1.addWidget(QLabel("x min")); hb1.addWidget(self.xmin)
        hb1.addWidget(QLabel("x max")); hb1.addWidget(self.xmax)
        hb2 = QHBoxLayout()
        hb2.addWidget(QLabel("y min")); hb2.addWidget(self.ymin)
        hb2.addWidget(QLabel("y max")); hb2.addWidget(self.ymax)

        # Buttons
        self.run_btn = QPushButton("Plot / Compute")
        self.clear_btn = QPushButton("Clear")

        bbar = QHBoxLayout()
        bbar.addWidget(self.run_btn)
        bbar.addWidget(self.clear_btn)

        # Plot + output
        self.canvas = MplCanvas()
        self.output = QTextEdit()
        self.output.setReadOnly(True)
        self.output.setPlaceholderText("Numerical output (radii, harmonics, etc.) will appear here...")

        # Layout
        top = QHBoxLayout()
        left = QVBoxLayout()
        left.addWidget(QLabel("Layer Arrays (input):"))
        left.addWidget(self.layers_edit)
        left.addLayout(form)
        left.addLayout(hb1)
        left.addLayout(hb2)
        left.addLayout(bbar)
        top.addLayout(left, 1)

        right = QVBoxLayout()
        right.addWidget(self.canvas, 3)
        right.addWidget(QLabel("Output:"))
        right.addWidget(self.output, 1)
        top.addLayout(right, 2)

        self.setLayout(top)
        self.resize(1200, 800)

        # Signals
        self.run_btn.clicked.connect(self.run_compute)
        self.clear_btn.clicked.connect(self.clear_all)

        # Constants
        self.mu0 = 4 * np.pi * 1e-7

    # --------------------- Core compute and plot ---------------------

    def run_compute(self):
        try:
            layers = self._parse_layers()
            n_max = int(self.nmax.text())
            aper_rad = float(self.aper_rad.text())
            dr = float(self.dr.text())
            t = float(self.t.text())
            ref_rad = float(self.ref_rad.text())
            I0 = float(self.I0.text())
            mag_type = self.mag_type.currentIndex() + 1
            surf_plot = self.surf_plot.isChecked()
            all_quads = self.all_quads.isChecked()

            # Bounds
            manual_bounds = self._parse_bounds()
            if manual_bounds is None:
                bounds = [[-3, 3], [-3, 3]] if all_quads else [[0, 3], [0, 3]]
            else:
                bounds = manual_bounds

        except Exception as e:
            self._err(f"Input parsing error: {e}")
            return

        mu0 = self.mu0
        init_rad = aper_rad + dr

        # ----- Geometry: build conductor positions from layers -----
        x_quad1, y_quad1 = [], []
        rads = []
        for i, layer in enumerate(layers):
            r_layer = init_rad + (2*i*dr) + (i+1)*t
            rads.append(r_layer)
            counts = layer[0]
            ang_ranges = layer[1]
            for j, ang_list in enumerate(ang_ranges):
                x, y = plotter(ang_list[0], ang_list[1], r_layer, counts[j])
                x_quad1.append(x)
                y_quad1.append(y)

        x_totq1 = np.concatenate(x_quad1) if len(x_quad1) else np.array([])
        y_totq1 = np.concatenate(y_quad1) if len(y_quad1) else np.array([])

        # Reflect into other quadrants
        x_totq2, y_totq2 = -x_totq1,  y_totq1
        x_totq3, y_totq3 = -x_totq1, -y_totq1
        x_totq4, y_totq4 =  x_totq1, -y_totq1

        x_tot = np.concatenate((x_totq1, x_totq2, x_totq3, x_totq4)) if x_totq1.size else np.array([])
        y_tot = np.concatenate((y_totq1, y_totq2, y_totq3, y_totq4)) if y_totq1.size else np.array([])

        # ----- Grid for field -----
        x = np.linspace(bounds[0][0], bounds[0][1], 250)
        y = np.linspace(bounds[1][0], bounds[1][1], 250)
        X, Y = np.meshgrid(x, y)
        Bx_total = np.zeros_like(X)
        By_total = np.zeros_like(Y)

        n_list = [0, 1, 2, 3, 4]
        B_main_index = mag_type - 1
        harms = np.zeros(len(n_list))

        # ----- Accumulate fields & harmonics -----
        self.canvas.ax.clear()
        for xa, ya in zip(x_tot, y_tot):
            phi = np.arctan2(ya, xa)
            a = np.hypot(xa, ya)
            I = I0 if np.cos(mag_type * phi) >= 0 else -I0  # sign by magnet type

            Bx, By = B_harm(X, Y, xa, ya, I, mu0, n_max)
            if not surf_plot:
                self.canvas.ax.scatter(xa, ya, c=("red" if I >= 0 else "blue"), s=10, zorder=3)

            Bx_total += Bx
            By_total += By

            for j, n_val in enumerate(n_list):
                harms[j] += Harmonic(ref_rad, a, phi, n_val, I, mu0)

        B_mag = np.hypot(Bx_total, By_total)

        # ----- Plot field into a fixed colorbar axis -----
        # Always clear the dedicated cax to avoid ghost space
        self.canvas.cax.cla()
        self.canvas.cax.set_visible(True)
        self.canvas.cax.set_frame_on(True)

        if surf_plot:
            mask = (X**2 + Y**2 >= aper_rad**2) & (X**2 + Y**2 <= (rads[-1]+dr)**2 if rads else True)
            B_mag_masked = np.where(mask, B_mag, np.nan)
            pcm = self.canvas.ax.pcolormesh(X, Y, B_mag_masked, shading='auto', cmap='viridis', zorder=0)
            self.fig_colorbar = self.canvas.fig.colorbar(pcm, cax=self.canvas.cax, label='|B| (T)')
        else:
            strm = self.canvas.ax.streamplot(X, Y, Bx_total, By_total, color=B_mag, cmap='viridis', density=3, zorder=0)
            self.fig_colorbar = self.canvas.fig.colorbar(strm.lines, cax=self.canvas.cax, label='|B| (T)')

        # Rings
        for r in rads:
            circ_plot(self.canvas.ax, r+dr)
            circ_plot(self.canvas.ax, r-dr)

        circ_plot(self.canvas.ax, aper_rad, color='k:')
        circ_plot(self.canvas.ax, ref_rad,  color='k:')

        self.canvas.ax.axhline(0, color='black', linewidth=1, zorder=1)
        self.canvas.ax.axvline(0, color='black', linewidth=1, zorder=1)
        self.canvas.ax.set_aspect('equal')
        self.canvas.ax.set_xlim(bounds[0][0], bounds[0][1])
        self.canvas.ax.set_ylim(bounds[1][0], bounds[1][1])
        self.canvas.ax.set_xlabel("x")
        self.canvas.ax.set_ylabel("y")
        self.canvas.ax.set_title(f"Field {'Surface' if surf_plot else 'Streamlines'} | mag type={mag_type}")

        # No tight_layout here — GridSpec already manages space cleanly
        self.canvas.draw()

        # ----- Outputs -----
        radii_str = ", ".join(f"{r:.2f}" for r in rads) if rads else "(none)"
        if harms[B_main_index] != 0:
            rel = harms / harms[B_main_index]
        else:
            rel = harms  # avoid div by zero

        B_main = harms[B_main_index] if harms[B_main_index] != 0 else 1.0
        units = (harms / B_main) * 1e4

        self.output.setPlainText(
            f"Radii of Conductor Layers: [{radii_str}]\n"
            f"Harmonics [n=1,dipole; n=2,quad; n=3,sext; n=4,oct; n=5,deca]:\n  {harms}\n"
            f"Relative to main (index {B_main_index}, magnet type={mag_type}):\n  {rel}\n"
            f"Harmonics in 10^4 units:\n  {units}\n"
            f"Bounds used: x[{bounds[0][0]}, {bounds[0][1]}], y[{bounds[1][0]}, {bounds[1][1]}]"
        )

    # --------------------- Utilities ---------------------

    def clear_all(self):
        # Clear plot and colorbar axes completely (no leftover space)
        self.canvas.ax.clear()
        self.canvas.cax.cla()
        self.canvas.cax.set_frame_on(False)
        self.canvas.cax.tick_params(left=False, labelleft=False)
        self.canvas.draw()
        self.output.clear()

    def _parse_layers(self):
        text = self.layers_edit.toPlainText().strip()
        if not text:
            raise ValueError("Layers input is empty.")
        layers = ast.literal_eval(text)
        # Expect list of entries: [ [counts], [ (ang1, ang2), ... ] ]
        for layer in layers:
            if not (isinstance(layer, list) and len(layer) == 2):
                raise ValueError("Each layer must be [counts_list, angle_pair_list].")
            counts, angs = layer
            if len(counts) != len(angs):
                raise ValueError("Each counts list must match the number of angle pairs.")
        return layers

    def _parse_bounds(self):
        fields = [self.xmin.text().strip(), self.xmax.text().strip(),
                  self.ymin.text().strip(), self.ymax.text().strip()]
        if any(f != "" for f in fields):
            try:
                xmin = float(fields[0]) if fields[0] != "" else -3
                xmax = float(fields[1]) if fields[1] != "" else 3
                ymin = float(fields[2]) if fields[2] != "" else -3
                ymax = float(fields[3]) if fields[3] != "" else 3
                return [[xmin, xmax], [ymin, ymax]]
            except ValueError:
                raise ValueError("Bounds must be numeric if provided.")
        return None

    def _err(self, msg):
        QMessageBox.critical(self, "Error", msg)


def main():
    app = QApplication(sys.argv)
    w = MagnetGUI()
    w.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


SystemExit: 0

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [2]:
import sys, ast
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, QFormLayout,
    QLineEdit, QTextEdit, QLabel, QPushButton, QCheckBox, QMessageBox,
    QComboBox, QTabWidget, QTableWidget, QTableWidgetItem, QToolButton,
    QAbstractItemView, QMenuBar, QAction, QProgressDialog
)
from PyQt5.QtCore import Qt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec


# ==================== Math / Physics helpers (from your script) ====================

def plotter(lower, upper, rad, point_num):
    phi_list = np.linspace(np.deg2rad(lower), np.deg2rad(upper), point_num)
    x = rad*np.cos(phi_list)
    y = rad*np.sin(phi_list)
    return x, y

def circ_plot(ax, rad, color='k--'):
    circ = np.linspace(0, 2*np.pi, 500)
    x = rad*np.cos(circ)
    y = rad*np.sin(circ)
    ax.plot(x, y, color, zorder=1)

def B_harm(x, y, x_a, y_a, I, mu0, n_max):
    a = np.hypot(x_a, y_a)
    r = np.hypot(x, y)
    phi = np.arctan2(y_a, x_a)
    theta = np.arctan2(y, x)
    ang = phi - theta
    r = np.where(r == 0, 1e-10, r)

    B_r_in = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(r / a)**(n - 1) * np.sin(n * ang) for n in range(1, n_max+1)], axis=0)
    B_theta_in = (-1)*((mu0 * I) / (2 * np.pi * a)) * np.sum([(r / a)**(n - 1) * np.cos(n * ang) for n in range(1, n_max+1)], axis=0)

    B_r_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(a / r)**(n + 1) * np.sin(n * ang) for n in range(0, n_max+1)], axis=0)
    B_theta_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum([(a / r)**(n + 1) * np.cos(n * ang) for n in range(0, n_max+1)], axis=0)

    B_r = np.where(r > a, B_r_out, B_r_in)
    B_theta = np.where(r > a, B_theta_out, B_theta_in)

    B_x = B_r * np.cos(theta) - B_theta * np.sin(theta)
    B_y = B_r * np.sin(theta) + B_theta * np.cos(theta)
    return B_x, B_y

def Harmonic(R0, a, phi, n, I_wire, mu0):
    # Returns Gauss contribution (~1e4 factor included)
    return (mu0*I_wire/2*np.pi) * (10**4) * ((R0 / a)**n) * np.cos((n + 1) * phi)


# ==================== Matplotlib Canvas ====================

class MplCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(7, 6), dpi=100)
        # Reserve a fixed colorbar axis using GridSpec
        gs = GridSpec(1, 2, width_ratios=[20, 1], figure=self.fig)
        self.ax = self.fig.add_subplot(gs[0, 0])
        self.cax = self.fig.add_subplot(gs[0, 1])
        # Start with an empty colorbar axis
        self.cax.set_visible(True)
        self.cax.set_frame_on(False)
        self.cax.tick_params(left=False, labelleft=False)
        super().__init__(self.fig)


# ==================== Layer Tab Widget ====================

class LayerTab(QWidget):
    """
    A per-layer editor with a small table:
    rows = segments; columns = [Count, Start°, End°]
    """
    def __init__(self, title_hint=None, preset=None, parent=None):
        super().__init__(parent)
        self.title_hint = title_hint or "Layer"
        layout = QVBoxLayout(self)

        # little help text
        layout.addWidget(QLabel("Segments (add rows as needed). Count must be an integer."))

        self.table = QTableWidget(0, 3, self)
        self.table.setHorizontalHeaderLabels(["Count", "Start°", "End°"])
        self.table.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.table.setEditTriggers(QAbstractItemView.AllEditTriggers)
        layout.addWidget(self.table)

        # Row controls
        rowbar = QHBoxLayout()
        add_row_btn = QToolButton()
        add_row_btn.setText("+ Row")
        del_row_btn = QToolButton()
        del_row_btn.setText("− Row")
        rowbar.addWidget(add_row_btn)
        rowbar.addWidget(del_row_btn)
        rowbar.addStretch(1)
        layout.addLayout(rowbar)

        add_row_btn.clicked.connect(self.add_row)
        del_row_btn.clicked.connect(self.del_row)

        # Optional preset
        if preset is not None:
            # preset format: ([counts...], [(start,end), ...])
            counts, angs = preset
            for c, (a1, a2) in zip(counts, angs):
                self.add_row(c, a1, a2)
        else:
            # Add a starter row
            self.add_row(3, 3, 13)

    def add_row(self, count=3, start=0.0, end=10.0):
        r = self.table.rowCount()
        self.table.insertRow(r)
        self.table.setItem(r, 0, QTableWidgetItem(str(int(count))))
        self.table.setItem(r, 1, QTableWidgetItem(str(float(start))))
        self.table.setItem(r, 2, QTableWidgetItem(str(float(end))))

    def del_row(self):
        rows = sorted(set(idx.row() for idx in self.table.selectedIndexes()), reverse=True)
        if not rows and self.table.rowCount() > 0:
            rows = [self.table.rowCount()-1]
        for r in rows:
            self.table.removeRow(r)

    def to_layer_struct(self):
        """
        Returns the layer in the expected format:
        [ [counts list], [ (ang1, ang2), ... ] ]
        """
        counts = []
        angs = []
        for r in range(self.table.rowCount()):
            c_item = self.table.item(r, 0)
            a1_item = self.table.item(r, 1)
            a2_item = self.table.item(r, 2)
            if c_item is None or a1_item is None or a2_item is None:
                continue
            try:
                c = int(float(c_item.text()))
                a1 = float(a1_item.text())
                a2 = float(a2_item.text())
            except ValueError:
                raise ValueError("Layer rows must have numeric Count/Start/End (Count = integer).")
            if c <= 0:
                raise ValueError("Counts must be positive.")
            counts.append(c)
            angs.append((a1, a2))
        if len(counts) != len(angs):
            raise ValueError("Mismatch in counts vs angle pairs on a layer tab.")
        if len(counts) == 0:
            raise ValueError("A layer tab is empty. Add at least one row or remove the layer.")
        return [counts, angs]

    def clone(self):
        return LayerTab(
            title_hint=self.title_hint,
            preset=self.to_layer_struct(),
            parent=self.parent()
        )


# ==================== Main GUI ====================

class MagnetGUI(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Magnet Field GUI (Tabbed Layers + Loading Indicator)")

        # ===== MENUBAR =====
        menubar = QMenuBar(self)
        layers_menu = menubar.addMenu("&Layers")

        act_add = QAction("Add Layer", self)
        act_dup = QAction("Duplicate Current Layer", self)
        act_del = QAction("Remove Current Layer", self)
        layers_menu.addAction(act_add)
        layers_menu.addAction(act_dup)
        layers_menu.addAction(act_del)

        tools_menu = menubar.addMenu("&Tools")
        act_import = QAction("Import Layers from Text", self)
        act_export = QAction("Export Layers to Text", self)
        tools_menu.addAction(act_import)
        tools_menu.addAction(act_export)

        # ===== LEFT SIDE CONTROLS =====
        self.layers_edit = QTextEdit()
        self.layers_edit.setPlaceholderText(
            "Optional raw layers text I/O.\n"
            "Format: [ [ [counts...], [(start,end), ...] ], ... ]\n"
            "Example: [ [ [3,6], [(3,13),(50,70)] ], [ [3,6], [(3,13),(50,70)] ] ]"
        )
        self.layers_edit.setVisible(False)  # hidden by default; used for Import/Export

        form = QFormLayout()
        self.nmax = QLineEdit("10")
        self.aper_rad = QLineEdit("2")
        self.dr = QLineEdit("0.1")
        self.t = QLineEdit("0.07")
        self.ref_rad = QLineEdit("1")
        self.I0 = QLineEdit("500")

        self.mag_type = QComboBox()
        self.mag_type.addItems(["Dipole (1)", "Quadrupole (2)", "Sextupole (3)"])
        self.mag_type.setCurrentIndex(2)  # Sextupole default

        self.surf_plot = QCheckBox("Surface plot (field magnitude)")
        self.surf_plot.setChecked(False)
        self.all_quads = QCheckBox("Show all quadrants")
        self.all_quads.setChecked(False)

        # Optional manual bounds
        self.xmin = QLineEdit("")
        self.xmax = QLineEdit("")
        self.ymin = QLineEdit("")
        self.ymax = QLineEdit("")

        form.addRow("n_max:", self.nmax)
        form.addRow("Aperture radius (aper_rad):", self.aper_rad)
        form.addRow("Radial spacing (dr):", self.dr)
        form.addRow("Layer thickness (t):", self.t)
        form.addRow("Reference radius (ref_rad):", self.ref_rad)
        form.addRow("Current magnitude (I0):", self.I0)
        form.addRow("Magnet type:", self.mag_type)
        form.addRow(self.surf_plot)
        form.addRow(self.all_quads)
        form.addRow(QLabel("Manual bounds (optional):"))
        hb1 = QHBoxLayout()
        hb1.addWidget(QLabel("x min")); hb1.addWidget(self.xmin)
        hb1.addWidget(QLabel("x max")); hb1.addWidget(self.xmax)
        hb2 = QHBoxLayout()
        hb2.addWidget(QLabel("y min")); hb2.addWidget(self.ymin)
        hb2.addWidget(QLabel("y max")); hb2.addWidget(self.ymax)

        # Buttons
        self.run_btn = QPushButton("Plot / Compute")
        self.clear_btn = QPushButton("Clear")

        bbar = QHBoxLayout()
        bbar.addWidget(self.run_btn)
        bbar.addWidget(self.clear_btn)

        # ===== LAYER TABS =====
        self.layer_tabs = QTabWidget()
        self.layer_tabs.setTabsClosable(True)
        self.layer_tabs.tabCloseRequested.connect(self._close_layer_tab)

        # default three example layers (sextupole style)
        example = ([3, 6], [(3, 13), (50, 70)])
        for i in range(3):
            self.add_layer_tab(preset=example)

        # ===== PLOT + OUTPUT =====
        self.canvas = MplCanvas()
        self.output = QTextEdit()
        self.output.setReadOnly(True)
        self.output.setPlaceholderText("Numerical output (radii, harmonics, etc.) will appear here...")

        # ===== LAYOUT =====
        outer = QVBoxLayout(self)
        outer.addWidget(menubar)

        top = QHBoxLayout()
        left = QVBoxLayout()
        left.addWidget(QLabel("Layer Tabs (edit segments per layer):"))
        left.addWidget(self.layer_tabs)
        left.addLayout(form)
        left.addLayout(hb1)
        left.addLayout(hb2)
        left.addLayout(bbar)
        left.addWidget(self.layers_edit)  # hidden unless user chooses Import/Export
        top.addLayout(left, 1)

        right = QVBoxLayout()
        right.addWidget(self.canvas, 3)
        right.addWidget(QLabel("Output:"))
        right.addWidget(self.output, 1)
        top.addLayout(right, 2)

        outer.addLayout(top)
        self.setLayout(outer)
        self.resize(1250, 820)

        # Signals
        self.run_btn.clicked.connect(self.run_compute)
        self.clear_btn.clicked.connect(self.clear_all)

        act_add.triggered.connect(lambda: self.add_layer_tab())
        act_dup.triggered.connect(self.duplicate_current_layer)
        act_del.triggered.connect(self.remove_current_layer)

        act_import.triggered.connect(self.import_layers_text)
        act_export.triggered.connect(self.export_layers_text)

        # Constants
        self.mu0 = 4 * np.pi * 1e-7

        # Loading indicator (initialized but hidden)
        self.spinner = None

    # --------------------- Layer helpers ---------------------

    def add_layer_tab(self, preset=None):
        idx = self.layer_tabs.count() + 1
        tab = LayerTab(title_hint=f"Layer {idx}", preset=preset, parent=self.layer_tabs)
        self.layer_tabs.addTab(tab, f"Layer {idx}")
        self.layer_tabs.setCurrentWidget(tab)

    def _close_layer_tab(self, index):
        if self.layer_tabs.count() <= 1:
            QMessageBox.warning(self, "Cannot Remove", "At least one layer must remain.")
            return
        self.layer_tabs.removeTab(index)
        self._renumber_tabs()

    def _renumber_tabs(self):
        for i in range(self.layer_tabs.count()):
            self.layer_tabs.setTabText(i, f"Layer {i+1}")

    def remove_current_layer(self):
        if self.layer_tabs.count() <= 1:
            QMessageBox.warning(self, "Cannot Remove", "At least one layer must remain.")
            return
        self.layer_tabs.removeTab(self.layer_tabs.currentIndex())
        self._renumber_tabs()

    def duplicate_current_layer(self):
        w = self.layer_tabs.currentWidget()
        if w is None:
            return
        try:
            clone = w.clone()
        except Exception as e:
            self._err(f"Could not duplicate layer: {e}")
            return
        idx = self.layer_tabs.count() + 1
        self.layer_tabs.addTab(clone, f"Layer {idx}")
        self.layer_tabs.setCurrentWidget(clone)

    def _collect_layers_from_tabs(self):
        layers = []
        for i in range(self.layer_tabs.count()):
            tab = self.layer_tabs.widget(i)
            layers.append(tab.to_layer_struct())
        return layers

    # --------------------- Import / Export raw text ---------------------

    def import_layers_text(self):
        self.layers_edit.setVisible(True)
        text = self.layers_edit.toPlainText().strip()
        if not text:
            QMessageBox.information(self, "Import", "Paste raw layers into the text box first, then click Import again.")
            return
        try:
            raw = ast.literal_eval(text)
            if not isinstance(raw, list):
                raise ValueError("Root must be a list of layers.")
            # Clear existing tabs
            while self.layer_tabs.count() > 0:
                self.layer_tabs.removeTab(0)
            for i, layer in enumerate(raw, start=1):
                if not (isinstance(layer, list) and len(layer) == 2):
                    raise ValueError("Each layer must be [counts_list, angle_pair_list].")
                counts, angs = layer
                if len(counts) != len(angs):
                    raise ValueError("Counts list must match angle pairs.")
                tab = LayerTab(title_hint=f"Layer {i}", preset=(counts, angs))
                self.layer_tabs.addTab(tab, f"Layer {i}")
            self.layer_tabs.setCurrentIndex(0)
            QMessageBox.information(self, "Import", "Layers imported into tabs.")
        except Exception as e:
            self._err(f"Import error: {e}")

    def export_layers_text(self):
        self.layers_edit.setVisible(True)
        try:
            layers = self._collect_layers_from_tabs()
            self.layers_edit.setPlainText(str(layers))
            QMessageBox.information(self, "Export", "Current tabs exported to the text box below.")
        except Exception as e:
            self._err(f"Export error: {e}")

    # --------------------- Core compute and plot ---------------------

    def run_compute(self):
        try:
            layers = self._collect_layers_from_tabs()
            n_max = int(self.nmax.text())
            aper_rad = float(self.aper_rad.text())
            dr = float(self.dr.text())
            t = float(self.t.text())
            ref_rad = float(self.ref_rad.text())
            I0 = float(self.I0.text())
            mag_type = self.mag_type.currentIndex() + 1
            surf_plot = self.surf_plot.isChecked()
            all_quads = self.all_quads.isChecked()

            # Bounds
            manual_bounds = self._parse_bounds()
            if manual_bounds is None:
                bounds = [[-3, 3], [-3, 3]] if all_quads else [[0, 3], [0, 3]]
            else:
                bounds = manual_bounds

        except Exception as e:
            self._err(f"Input parsing error: {e}")
            return

        mu0 = self.mu0
        init_rad = aper_rad + dr

        # ----- Loading Spinner -----
        self._start_spinner("Computing field...")

        try:
            # ----- Geometry: build conductor positions from layers -----
            x_quad1, y_quad1 = [], []
            rads = []
            for i, layer in enumerate(layers):
                # keep UI responsive as we build geometry
                QApplication.processEvents()
                r_layer = init_rad + (2*i*dr) + (i+1)*t
                rads.append(r_layer)
                counts = layer[0]
                ang_ranges = layer[1]
                for j, ang_list in enumerate(ang_ranges):
                    x, y = plotter(ang_list[0], ang_list[1], r_layer, counts[j])
                    x_quad1.append(x)
                    y_quad1.append(y)

            x_totq1 = np.concatenate(x_quad1) if len(x_quad1) else np.array([])
            y_totq1 = np.concatenate(y_quad1) if len(y_quad1) else np.array([])

            # Reflect into other quadrants
            x_totq2, y_totq2 = -x_totq1,  y_totq1
            x_totq3, y_totq3 = -x_totq1, -y_totq1
            x_totq4, y_totq4 =  x_totq1, -y_totq1

            x_tot = np.concatenate((x_totq1, x_totq2, x_totq3, x_totq4)) if x_totq1.size else np.array([])
            y_tot = np.concatenate((y_totq1, y_totq2, y_totq3, y_totq4)) if y_totq1.size else np.array([])

            # ----- Grid for field -----
            x = np.linspace(bounds[0][0], bounds[0][1], 250)
            y = np.linspace(bounds[1][0], bounds[1][1], 250)
            X, Y = np.meshgrid(x, y)
            Bx_total = np.zeros_like(X)
            By_total = np.zeros_like(Y)

            n_list = [0, 1, 2, 3, 4]
            B_main_index = mag_type - 1
            harms = np.zeros(len(n_list))

            # ----- Accumulate fields & harmonics -----
            self.canvas.ax.clear()

            # progress pulses during compute
            total_pts = max(1, len(x_tot))
            for k, (xa, ya) in enumerate(zip(x_tot, y_tot), start=1):
                phi = np.arctan2(ya, xa)
                a = np.hypot(xa, ya)
                I = I0 if np.cos(mag_type * phi) >= 0 else -I0  # sign by magnet type

                Bx, By = B_harm(X, Y, xa, ya, I, mu0, n_max)
                if not surf_plot:
                    self.canvas.ax.scatter(xa, ya, c=("red" if I >= 0 else "blue"), s=10, zorder=3)

                Bx_total += Bx
                By_total += By

                for j, n_val in enumerate(n_list):
                    harms[j] += Harmonic(ref_rad, a, phi, n_val, I, mu0)

                # keep UI snappy & spinner alive
                if k % 25 == 0:
                    QApplication.processEvents()

            B_mag = np.hypot(Bx_total, By_total)

            # ----- Plot field into a fixed colorbar axis -----
            self.canvas.cax.cla()
            self.canvas.cax.set_visible(True)
            self.canvas.cax.set_frame_on(True)

            if surf_plot:
                mask = (X**2 + Y**2 >= aper_rad**2) & (X**2 + Y**2 <= (rads[-1]+dr)**2 if rads else True)
                B_mag_masked = np.where(mask, B_mag, np.nan)
                pcm = self.canvas.ax.pcolormesh(X, Y, B_mag_masked, shading='auto', cmap='viridis', zorder=0)
                self.fig_colorbar = self.canvas.fig.colorbar(pcm, cax=self.canvas.cax, label='|B| (T)')
            else:
                strm = self.canvas.ax.streamplot(X, Y, Bx_total, By_total, color=B_mag, cmap='viridis', density=3, zorder=0)
                self.fig_colorbar = self.canvas.fig.colorbar(strm.lines, cax=self.canvas.cax, label='|B| (T)')

            # Rings
            for r in rads:
                circ_plot(self.canvas.ax, r+dr)
                circ_plot(self.canvas.ax, r-dr)

            circ_plot(self.canvas.ax, aper_rad, color='k:')
            circ_plot(self.canvas.ax, ref_rad,  color='k:')

            self.canvas.ax.axhline(0, color='black', linewidth=1, zorder=1)
            self.canvas.ax.axvline(0, color='black', linewidth=1, zorder=1)
            self.canvas.ax.set_aspect('equal')
            self.canvas.ax.set_xlim(bounds[0][0], bounds[0][1])
            self.canvas.ax.set_ylim(bounds[1][0], bounds[1][1])
            self.canvas.ax.set_xlabel("x")
            self.canvas.ax.set_ylabel("y")
            self.canvas.ax.set_title(f"Field {'Surface' if surf_plot else 'Streamlines'} | mag type={mag_type}")

            self.canvas.draw()

            # ----- Outputs -----
            radii_str = ", ".join(f"{r:.2f}" for r in rads) if rads else "(none)"
            if harms[B_main_index] != 0:
                rel = harms / harms[B_main_index]
            else:
                rel = harms  # avoid div by zero

            B_main = harms[B_main_index] if harms[B_main_index] != 0 else 1.0
            units = (harms / B_main) * 1e4

            self.output.setPlainText(
                f"Radii of Conductor Layers: [{radii_str}]\n"
                f"Harmonics [n=1,dipole; n=2,quad; n=3,sext; n=4,oct; n=5,deca]:\n  {harms}\n"
                f"Relative to main (index {B_main_index}, magnet type={mag_type}):\n  {rel}\n"
                f"Harmonics in 10^4 units:\n  {units}\n"
                f"Bounds used: x[{bounds[0][0]}, {bounds[0][1]}], y[{bounds[1][0]}, {bounds[1][1]}]"
            )

        except Exception as e:
            self._err(f"Compute error: {e}")
        finally:
            self._stop_spinner()

    # --------------------- Utilities ---------------------

    def clear_all(self):
        # Clear plot and colorbar axes completely (no leftover space)
        self.canvas.ax.clear()
        self.canvas.cax.cla()
        self.canvas.cax.set_frame_on(False)
        self.canvas.cax.tick_params(left=False, labelleft=False)
        self.canvas.draw()
        self.output.clear()

    def _parse_bounds(self):
        fields = [self.xmin.text().strip(), self.xmax.text().strip(),
                  self.ymin.text().strip(), self.ymax.text().strip()]
        if any(f != "" for f in fields):
            try:
                xmin = float(fields[0]) if fields[0] != "" else -3
                xmax = float(fields[1]) if fields[1] != "" else 3
                ymin = float(fields[2]) if fields[2] != "" else -3
                ymax = float(fields[3]) if fields[3] != "" else 3
                return [[xmin, xmax], [ymin, ymax]]
            except ValueError:
                raise ValueError("Bounds must be numeric if provided.")
        return None

    def _err(self, msg):
        QMessageBox.critical(self, "Error", msg)

    # ----- Loading spinner helpers -----
    def _start_spinner(self, text="Working..."):
        self.spinner = QProgressDialog(text, None, 0, 0, self,
                                       flags=Qt.WindowTitleHint | Qt.CustomizeWindowHint)
        self.spinner.setWindowModality(Qt.ApplicationModal)
        self.spinner.setAutoClose(False)
        self.spinner.setCancelButton(None)
        self.spinner.setMinimumDuration(0)
        self.spinner.setRange(0, 0)  # busy indicator
        self.spinner.show()
        QApplication.processEvents()

    def _stop_spinner(self):
        if self.spinner is not None:
            self.spinner.close()
            self.spinner = None


def main():
    app = QApplication(sys.argv)
    w = MagnetGUI()
    w.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


SystemExit: 0

In [3]:
# magnet_gui.py
# Requires: PyQt5, matplotlib, numpy
# Run: python magnet_gui.py

import sys, ast
import numpy as np
from PyQt5.QtWidgets import (
    QApplication, QWidget, QVBoxLayout, QHBoxLayout, QFormLayout,
    QLineEdit, QTextEdit, QLabel, QPushButton, QCheckBox, QMessageBox,
    QComboBox, QTabWidget, QTableWidget, QTableWidgetItem, QToolButton,
    QAbstractItemView, QMenuBar, QAction, QProgressDialog
)
from PyQt5.QtCore import Qt
from matplotlib.backends.backend_qt5agg import FigureCanvasQTAgg as FigureCanvas
from matplotlib.figure import Figure
from matplotlib.gridspec import GridSpec


# ==================== Physics / math helpers ====================

def plotter(lower, upper, rad, point_num):
    phi_list = np.linspace(np.deg2rad(lower), np.deg2rad(upper), point_num)
    x = rad * np.cos(phi_list)
    y = rad * np.sin(phi_list)
    return x, y

def circ_plot(ax, rad, color='k--'):
    circ = np.linspace(0, 2*np.pi, 500)
    x = rad * np.cos(circ)
    y = rad * np.sin(circ)
    ax.plot(x, y, color, zorder=1)

def B_harm(x, y, x_a, y_a, I, mu0, n_max):
    """
    Field from a single conductor using multipole expansion
    """
    a = np.hypot(x_a, y_a)         # radius of conductor
    r = np.hypot(x, y)             # radius of eval point
    phi = np.arctan2(y_a, x_a)     # conductor angle
    theta = np.arctan2(y, x)       # eval point angle
    ang = phi - theta

    # avoid div by 0 at origin
    r = np.where(r == 0, 1e-10, r)

    # interior expansion
    B_r_in = ((mu0 * I) / (2 * np.pi * a)) * np.sum(
        [(r / a)**(n - 1) * np.sin(n * ang) for n in range(1, n_max+1)],
        axis=0
    )
    B_theta_in = (-1) * ((mu0 * I) / (2 * np.pi * a)) * np.sum(
        [(r / a)**(n - 1) * np.cos(n * ang) for n in range(1, n_max+1)],
        axis=0
    )

    # exterior expansion
    B_r_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum(
        [(a / r)**(n + 1) * np.sin(n * ang) for n in range(0, n_max+1)],
        axis=0
    )
    B_theta_out = ((mu0 * I) / (2 * np.pi * a)) * np.sum(
        [(a / r)**(n + 1) * np.cos(n * ang) for n in range(0, n_max+1)],
        axis=0
    )

    # pick correct region
    B_r = np.where(r > a, B_r_out, B_r_in)
    B_theta = np.where(r > a, B_theta_out, B_theta_in)

    # convert polar -> cartesian
    B_x = B_r * np.cos(theta) - B_theta * np.sin(theta)
    B_y = B_r * np.sin(theta) + B_theta * np.cos(theta)
    return B_x, B_y

def Harmonic(R0, a, phi, n, I_wire, mu0):
    """
    Harmonic contribution in Gauss units-ish (10^4 factor).
    n=0 -> dipole-like, n=1 -> quad-like, etc.
    """
    return (mu0 * I_wire / (2 * np.pi)) * (10**4) * ((R0 / a)**n) * np.cos((n + 1) * phi)


# ==================== Matplotlib Canvas widget ====================

class MplCanvas(FigureCanvas):
    def __init__(self):
        self.fig = Figure(figsize=(7, 6), dpi=100)

        # We manage layout explicitly: plot axis + dedicated colorbar axis
        gs = GridSpec(1, 2, width_ratios=[20, 1], figure=self.fig)
        self.ax = self.fig.add_subplot(gs[0, 0])
        self.cax = self.fig.add_subplot(gs[0, 1])

        # Set up the colorbar axis initially blank
        self.cax.set_visible(True)
        self.cax.set_frame_on(False)
        self.cax.tick_params(left=False, labelleft=False)

        super().__init__(self.fig)


# ==================== LayerTab: editable per-layer config ====================

class LayerTab(QWidget):
    """
    Each layer tab describes ONE radial layer of conductors:
      - multiple angular segments
      - each row in the table = one angular block
        Count  Start°  End°
    We later turn this into the format you were using:
        [ [counts...], [(start,end), ...] ]
    """

    def __init__(self, title_hint=None, preset=None, parent=None):
        super().__init__(parent)
        self.title_hint = title_hint or "Layer"

        layout = QVBoxLayout(self)

        layout.addWidget(QLabel("Segments for this layer (Count = number of conductors on that arc):"))

        # table: rows = segments
        self.table = QTableWidget(0, 3, self)
        self.table.setHorizontalHeaderLabels(["Count", "Start°", "End°"])
        self.table.setSelectionBehavior(QAbstractItemView.SelectRows)
        self.table.setEditTriggers(QAbstractItemView.AllEditTriggers)
        layout.addWidget(self.table)

        # mini row control bar
        rowbar = QHBoxLayout()
        self.btn_add_row = QToolButton()
        self.btn_add_row.setText("+ Row")
        self.btn_del_row = QToolButton()
        self.btn_del_row.setText("− Row")
        rowbar.addWidget(self.btn_add_row)
        rowbar.addWidget(self.btn_del_row)
        rowbar.addStretch(1)
        layout.addLayout(rowbar)

        self.btn_add_row.clicked.connect(self.add_row)
        self.btn_del_row.clicked.connect(self.del_row)

        # optional prefill
        if preset is not None:
            counts, angs = preset
            for c, (a1, a2) in zip(counts, angs):
                self.add_row(c, a1, a2)
        else:
            # add a starter row so it isn't empty
            self.add_row(3, 3, 13)

    def add_row(self, count=3, start=0.0, end=10.0):
        r = self.table.rowCount()
        self.table.insertRow(r)
        self.table.setItem(r, 0, QTableWidgetItem(str(int(count))))
        self.table.setItem(r, 1, QTableWidgetItem(str(float(start))))
        self.table.setItem(r, 2, QTableWidgetItem(str(float(end))))

    def del_row(self):
        rows = sorted({idx.row() for idx in self.table.selectedIndexes()}, reverse=True)
        # if no selection, delete last row (if exists)
        if not rows and self.table.rowCount() > 0:
            rows = [self.table.rowCount()-1]
        for r in rows:
            self.table.removeRow(r)

    def to_layer_struct(self):
        """
        Return this layer in code-friendly format:
            [ [counts list], [ (ang1, ang2), ... ] ]
        For example:
            [[3,6], [(3,13),(50,70)]]
        """
        counts = []
        angs = []
        for r in range(self.table.rowCount()):
            c_item = self.table.item(r, 0)
            a1_item = self.table.item(r, 1)
            a2_item = self.table.item(r, 2)

            if c_item is None or a1_item is None or a2_item is None:
                continue

            try:
                c = int(float(c_item.text()))
                a1 = float(a1_item.text())
                a2 = float(a2_item.text())
            except ValueError:
                raise ValueError("Layer rows must have numeric Count / Start° / End°. Count must be integer.")

            if c <= 0:
                raise ValueError("Counts must be positive.")

            counts.append(c)
            angs.append((a1, a2))

        if len(counts) == 0:
            raise ValueError("A layer tab is empty. Add at least one segment row or remove the layer.")

        if len(counts) != len(angs):
            raise ValueError("Mismatch in this layer between count list and angle list.")

        return [counts, angs]

    def clone(self):
        """
        Returns a brand new LayerTab with same rows.
        """
        return LayerTab(
            title_hint=self.title_hint,
            preset=self.to_layer_struct(),
            parent=self.parent()
        )


# ==================== Main GUI ====================

class MagnetGUI(QWidget):
    def __init__(self):
        super().__init__()
        self.setWindowTitle("Magnet Field GUI (Tabbed Layers + Spinner)")

        # ===== MENU BAR =====
        menubar = QMenuBar(self)
        layers_menu = menubar.addMenu("&Layers")
        act_add_layer = QAction("Add Layer", self)
        act_dup_layer = QAction("Duplicate Current Layer", self)
        act_del_layer = QAction("Remove Current Layer", self)
        layers_menu.addAction(act_add_layer)
        layers_menu.addAction(act_dup_layer)
        layers_menu.addAction(act_del_layer)

        tools_menu = menubar.addMenu("&Tools")
        act_import = QAction("Import Layers from Text", self)
        act_export = QAction("Export Layers to Text", self)
        tools_menu.addAction(act_import)
        tools_menu.addAction(act_export)

        # ===== LAYER TABS + 'Add Layer' BUTTON =====
        self.layer_tabs = QTabWidget()
        self.layer_tabs.setTabsClosable(True)
        self.layer_tabs.tabCloseRequested.connect(self._close_layer_tab)

        # Explicit button under tabs
        self.btn_add_layer = QPushButton("＋ Add Layer")
        self.btn_add_layer.clicked.connect(lambda: self.add_layer_tab())

        # ===== OPTIONAL RAW TEXT BOX (hidden normally) =====
        self.layers_edit = QTextEdit()
        self.layers_edit.setPlaceholderText(
            "Optional raw layers text I/O.\n"
            "Format: [ [ [counts...], [(start,end), ...] ], ... ]\n"
            "Example:\n"
            "[ [[3,6], [(3,13),(50,70)]], [[3,6], [(3,13),(50,70)]] ]"
        )
        self.layers_edit.setVisible(False)

        # ===== PHYSICS / RUNTIME PARAM FORM =====
        form = QFormLayout()
        self.nmax     = QLineEdit("10")
        self.aper_rad = QLineEdit("2")
        self.dr       = QLineEdit("0.1")
        self.t        = QLineEdit("0.07")
        self.ref_rad  = QLineEdit("1")
        self.I0       = QLineEdit("500")

        self.mag_type = QComboBox()
        self.mag_type.addItems(["Dipole (1)", "Quadrupole (2)", "Sextupole (3)"])
        self.mag_type.setCurrentIndex(2)  # sextupole default

        self.surf_plot = QCheckBox("Surface plot (field magnitude)")
        self.surf_plot.setChecked(False)
        self.all_quads = QCheckBox("Show all quadrants")
        self.all_quads.setChecked(False)

        # plot bounds (optional override)
        self.xmin = QLineEdit("")
        self.xmax = QLineEdit("")
        self.ymin = QLineEdit("")
        self.ymax = QLineEdit("")

        form.addRow("n_max:", self.nmax)
        form.addRow("Aperture radius (aper_rad):", self.aper_rad)
        form.addRow("Radial spacing (dr):", self.dr)
        form.addRow("Layer thickness (t):", self.t)
        form.addRow("Reference radius (ref_rad):", self.ref_rad)
        form.addRow("Current magnitude (I0):", self.I0)
        form.addRow("Magnet type:", self.mag_type)
        form.addRow(self.surf_plot)
        form.addRow(self.all_quads)

        # manual bounds rows
        bounds_label = QLabel("Manual bounds (optional):")
        hb1 = QHBoxLayout()
        hb1.addWidget(QLabel("x min")); hb1.addWidget(self.xmin)
        hb1.addWidget(QLabel("x max")); hb1.addWidget(self.xmax)
        hb2 = QHBoxLayout()
        hb2.addWidget(QLabel("y min")); hb2.addWidget(self.ymin)
        hb2.addWidget(QLabel("y max")); hb2.addWidget(self.ymax)

        # ===== RUN / CLEAR BUTTONS =====
        self.run_btn = QPushButton("Plot / Compute")
        self.clear_btn = QPushButton("Clear")
        bbar = QHBoxLayout()
        bbar.addWidget(self.run_btn)
        bbar.addWidget(self.clear_btn)

        # ===== MATPLOTLIB CANVAS + OUTPUT BOX =====
        self.canvas = MplCanvas()
        self.output = QTextEdit()
        self.output.setReadOnly(True)
        self.output.setPlaceholderText("Numerical output (radii, harmonics, etc.) will appear here...")

        # ===== LAYOUT STRUCTURE =====
        outer = QVBoxLayout(self)
        outer.addWidget(menubar)

        main_hbox = QHBoxLayout()

        # LEFT PANE
        left = QVBoxLayout()
        left.addWidget(QLabel("Layer Tabs (edit segments per layer):"))
        left.addWidget(self.layer_tabs)
        left.addWidget(self.btn_add_layer)  # <-- Add Layer button visible in UI
        left.addLayout(form)
        left.addWidget(bounds_label)
        left.addLayout(hb1)
        left.addLayout(hb2)
        left.addLayout(bbar)
        left.addWidget(self.layers_edit)  # stays hidden unless Import/Export
        main_hbox.addLayout(left, 1)

        # RIGHT PANE
        right = QVBoxLayout()
        right.addWidget(self.canvas, 3)
        right.addWidget(QLabel("Output:"))
        right.addWidget(self.output, 1)
        main_hbox.addLayout(right, 2)

        outer.addLayout(main_hbox)

        self.setLayout(outer)
        self.resize(1250, 820)

        # ===== SIGNALS =====
        self.run_btn.clicked.connect(self.run_compute)
        self.clear_btn.clicked.connect(self.clear_all)

        act_add_layer.triggered.connect(lambda: self.add_layer_tab())
        act_dup_layer.triggered.connect(self.duplicate_current_layer)
        act_del_layer.triggered.connect(self.remove_current_layer)

        act_import.triggered.connect(self.import_layers_text)
        act_export.triggered.connect(self.export_layers_text)

        # ===== CONSTANTS =====
        self.mu0 = 4 * np.pi * 1e-7

        # ===== LOADING / SPINNER DIALOG HANDLE =====
        self.spinner = None

        # ===== CREATE INITIAL DEFAULT LAYERS =====
        # We'll give you 3 starter layers ~sextupole style:
        # counts [3,6], ang ranges [(3,13),(50,70)]
        default_layer = ([3, 6], [(3, 13), (50, 70)])
        for _ in range(3):
            self.add_layer_tab(preset=default_layer)

    # --------------------- Layer management ---------------------

    def add_layer_tab(self, preset=None):
        """
        Create a new LayerTab and add as new tab.
        """
        next_idx = self.layer_tabs.count() + 1
        tab = LayerTab(title_hint=f"Layer {next_idx}", preset=preset, parent=self.layer_tabs)
        self.layer_tabs.addTab(tab, f"Layer {next_idx}")
        self.layer_tabs.setCurrentWidget(tab)

    def _close_layer_tab(self, index):
        """
        Closes layer tab via [x] on tab bar.
        Keep at least one tab always.
        """
        if self.layer_tabs.count() <= 1:
            QMessageBox.warning(self, "Cannot Remove", "At least one layer must remain.")
            return
        self.layer_tabs.removeTab(index)
        self._renumber_tabs()

    def _renumber_tabs(self):
        for i in range(self.layer_tabs.count()):
            self.layer_tabs.setTabText(i, f"Layer {i+1}")

    def remove_current_layer(self):
        """
        Menu: Remove Current Layer
        """
        if self.layer_tabs.count() <= 1:
            QMessageBox.warning(self, "Cannot Remove", "At least one layer must remain.")
            return
        idx = self.layer_tabs.currentIndex()
        if idx >= 0:
            self.layer_tabs.removeTab(idx)
            self._renumber_tabs()

    def duplicate_current_layer(self):
        """
        Menu: Duplicate Current Layer
        """
        w = self.layer_tabs.currentWidget()
        if w is None:
            return
        try:
            clone = w.clone()
        except Exception as e:
            self._err(f"Could not duplicate layer: {e}")
            return

        next_idx = self.layer_tabs.count() + 1
        self.layer_tabs.addTab(clone, f"Layer {next_idx}")
        self.layer_tabs.setCurrentWidget(clone)

    def _collect_layers_from_tabs(self):
        """
        Returns the full magnet 'layers' array from all tabs:
        [
          [ [counts...], [(a1,a2), ...] ],
          [ [counts...], [(a1,a2), ...] ],
          ...
        ]
        """
        out = []
        for i in range(self.layer_tabs.count()):
            tab = self.layer_tabs.widget(i)
            out.append(tab.to_layer_struct())
        return out

    # --------------------- Import / Export ---------------------

    def import_layers_text(self):
        """
        Read self.layers_edit text, parse as python literal into tabs.
        """
        # show the box if it's hidden
        self.layers_edit.setVisible(True)

        raw_txt = self.layers_edit.toPlainText().strip()
        if not raw_txt:
            QMessageBox.information(
                self,
                "Import",
                "Paste your raw layers list into the text box first, then run Import again."
            )
            return

        try:
            parsed = ast.literal_eval(raw_txt)
            if not isinstance(parsed, list):
                raise ValueError("Top level must be a list of layers.")

            # Clear current tabs
            while self.layer_tabs.count() > 0:
                self.layer_tabs.removeTab(0)

            # Build tabs from imported data
            for i, layer in enumerate(parsed, start=1):
                if not (isinstance(layer, list) and len(layer) == 2):
                    raise ValueError("Each layer must look like [counts_list, angle_pair_list].")

                counts, angs = layer
                if len(counts) != len(angs):
                    raise ValueError("counts_list and angle_pair_list length mismatch.")

                tab = LayerTab(title_hint=f"Layer {i}", preset=(counts, angs), parent=self.layer_tabs)
                self.layer_tabs.addTab(tab, f"Layer {i}")

            self.layer_tabs.setCurrentIndex(0)
            QMessageBox.information(self, "Import", "Layers imported into tabs.")

        except Exception as e:
            self._err(f"Import error: {e}")

    def export_layers_text(self):
        """
        Dump current tab state back into self.layers_edit as text.
        """
        self.layers_edit.setVisible(True)
        try:
            layers = self._collect_layers_from_tabs()
            self.layers_edit.setPlainText(str(layers))
            QMessageBox.information(self, "Export", "Current tabs exported below.")
        except Exception as e:
            self._err(f"Export error: {e}")

    # --------------------- Compute / Plot ---------------------

    def run_compute(self):
        """
        Core compute button:
        - read GUI state
        - build conductor positions
        - compute full field / colorbar / harmonics
        - update text output
        Includes a spinner to show it's working.
        """
        try:
            layers = self._collect_layers_from_tabs()

            n_max     = int(self.nmax.text())
            aper_rad  = float(self.aper_rad.text())
            dr        = float(self.dr.text())
            t         = float(self.t.text())
            ref_rad   = float(self.ref_rad.text())
            I0        = float(self.I0.text())
            mag_type  = self.mag_type.currentIndex() + 1
            surf_plot = self.surf_plot.isChecked()
            all_quads = self.all_quads.isChecked()

            manual_bounds = self._parse_bounds()
            if manual_bounds is None:
                bounds = [[-3, 3], [-3, 3]] if all_quads else [[0, 3], [0, 3]]
            else:
                bounds = manual_bounds

        except Exception as e:
            self._err(f"Input parsing error: {e}")
            return

        mu0 = self.mu0
        init_rad = aper_rad + dr

        # show spinner while computing
        self._start_spinner("Computing field...")

        try:
            # ----- Build conductor coordinates layer by layer -----
            x_quad1, y_quad1 = [], []
            rads = []
            for i, layer in enumerate(layers):
                QApplication.processEvents()  # keep UI alive
                r_layer = init_rad + (2*i*dr) + (i+1)*t  # radial center of this layer
                rads.append(r_layer)

                counts = layer[0]
                ang_ranges = layer[1]

                for j, ang_pair in enumerate(ang_ranges):
                    start_deg, end_deg = ang_pair
                    x_seg, y_seg = plotter(start_deg, end_deg, r_layer, counts[j])
                    x_quad1.append(x_seg)
                    y_quad1.append(y_seg)

            x_totq1 = np.concatenate(x_quad1) if x_quad1 else np.array([])
            y_totq1 = np.concatenate(y_quad1) if y_quad1 else np.array([])

            # mirror into 4 quads
            x_totq2, y_totq2 = -x_totq1,  y_totq1
            x_totq3, y_totq3 = -x_totq1, -y_totq1
            x_totq4, y_totq4 =  x_totq1, -y_totq1

            if x_totq1.size:
                x_tot = np.concatenate((x_totq1, x_totq2, x_totq3, x_totq4))
                y_tot = np.concatenate((y_totq1, y_totq2, y_totq3, y_totq4))
            else:
                x_tot = np.array([])
                y_tot = np.array([])

            # ----- Field grid -----
            xg = np.linspace(bounds[0][0], bounds[0][1], 250)
            yg = np.linspace(bounds[1][0], bounds[1][1], 250)
            X, Y = np.meshgrid(xg, yg)
            Bx_total = np.zeros_like(X)
            By_total = np.zeros_like(Y)

            # multipole bookkeeping
            n_list = [0, 1, 2, 3, 4]  # dipole..decapole-ish
            B_main_index = mag_type - 1
            harms = np.zeros(len(n_list))

            # ----- Sum contributions -----
            self.canvas.ax.clear()
            total_pts = max(1, len(x_tot))

            for k, (xa, ya) in enumerate(zip(x_tot, y_tot), start=1):
                phi = np.arctan2(ya, xa)
                a   = np.hypot(xa, ya)

                # sign of current depends on magnet type and coil azimuth
                I = I0 if np.cos(mag_type * phi) >= 0 else -I0

                Bx, By = B_harm(X, Y, xa, ya, I, mu0, n_max)

                if not surf_plot:
                    self.canvas.ax.scatter(
                        xa, ya,
                        c=("red" if I >= 0 else "blue"),
                        s=10,
                        zorder=3,
                    )

                Bx_total += Bx
                By_total += By

                # accumulate harmonic contributions
                for j, n_val in enumerate(n_list):
                    harms[j] += Harmonic(ref_rad, a, phi, n_val, I, mu0)

                # periodically keep spinner responsive
                if k % 25 == 0:
                    QApplication.processEvents()

            B_mag = np.hypot(Bx_total, By_total)

            # ----- Draw field + colorbar in reserved cax -----
            self.canvas.cax.cla()
            self.canvas.cax.set_visible(True)
            self.canvas.cax.set_frame_on(True)

            if surf_plot:
                # mask outside aperture and outside last layer
                mask = (X**2 + Y**2 >= aper_rad**2)
                if rads:
                    mask &= (X**2 + Y**2 <= (rads[-1] + dr)**2)
                B_mag_masked = np.where(mask, B_mag, np.nan)

                pcm = self.canvas.ax.pcolormesh(
                    X, Y,
                    B_mag_masked,
                    shading='auto',
                    cmap='viridis',
                    zorder=0
                )
                self.canvas.fig.colorbar(pcm, cax=self.canvas.cax, label='|B| (T)')

            else:
                strm = self.canvas.ax.streamplot(
                    X, Y,
                    Bx_total, By_total,
                    color=B_mag,
                    cmap='viridis',
                    density=3,
                    zorder=0
                )
                self.canvas.fig.colorbar(strm.lines, cax=self.canvas.cax, label='|B| (T)')

            # draw conductor ring outlines
            for r in rads:
                circ_plot(self.canvas.ax, r + dr)
                circ_plot(self.canvas.ax, r - dr)

            circ_plot(self.canvas.ax, float(self.aper_rad.text()), color='k:')
            circ_plot(self.canvas.ax, float(self.ref_rad.text()),  color='k:')

            # axes lines
            self.canvas.ax.axhline(0, color='black', linewidth=1, zorder=1)
            self.canvas.ax.axvline(0, color='black', linewidth=1, zorder=1)

            self.canvas.ax.set_aspect('equal')
            self.canvas.ax.set_xlim(bounds[0][0], bounds[0][1])
            self.canvas.ax.set_ylim(bounds[1][0], bounds[1][1])
            self.canvas.ax.set_xlabel("x")
            self.canvas.ax.set_ylabel("y")
            self.canvas.ax.set_title(
                f"Field {'Surface' if surf_plot else 'Streamlines'} | mag type={mag_type}"
            )

            self.canvas.draw()

            # ----- Output text (harmonics etc.) -----
            radii_str = ", ".join(f"{r:.2f}" for r in rads) if rads else "(none)"

            # relative harmonics vs main
            if harms[B_main_index] != 0:
                rel = harms / harms[B_main_index]
            else:
                rel = harms

            B_main = harms[B_main_index] if harms[B_main_index] != 0 else 1.0
            units = (harms / B_main) * 1e4

            self.output.setPlainText(
                f"Radii of Conductor Layers: [{radii_str}]\n"
                f"Harmonics [n=1,dipole; n=2,quad; n=3,sext; n=4,oct; n=5,deca]:\n  {harms}\n"
                f"Relative to main (index {B_main_index}, magnet type={mag_type}):\n  {rel}\n"
                f"Harmonics in 10^4 units:\n  {units}\n"
                f"Bounds used: x[{bounds[0][0]}, {bounds[0][1]}], "
                f"y[{bounds[1][0]}, {bounds[1][1]}]"
            )

        except Exception as e:
            self._err(f"Compute error: {e}")
        finally:
            self._stop_spinner()

    # --------------------- Misc utils ---------------------

    def clear_all(self):
        """
        Clear plot, colorbar, and output box.
        """
        self.canvas.ax.clear()
        self.canvas.cax.cla()
        self.canvas.cax.set_frame_on(False)
        self.canvas.cax.tick_params(left=False, labelleft=False)
        self.canvas.draw()
        self.output.clear()

    def _parse_bounds(self):
        """
        Read manual bounds if present, otherwise return None.
        Returns [[xmin,xmax],[ymin,ymax]].
        """
        fields = [
            self.xmin.text().strip(), self.xmax.text().strip(),
            self.ymin.text().strip(), self.ymax.text().strip()
        ]
        if any(f != "" for f in fields):
            try:
                xmin = float(fields[0]) if fields[0] != "" else -3
                xmax = float(fields[1]) if fields[1] != "" else  3
                ymin = float(fields[2]) if fields[2] != "" else -3
                ymax = float(fields[3]) if fields[3] != "" else  3
                return [[xmin, xmax], [ymin, ymax]]
            except ValueError:
                raise ValueError("Bounds must be numeric if provided.")
        return None

    def _err(self, msg):
        QMessageBox.critical(self, "Error", msg)

    # ---------- Spinner helpers ----------
    def _start_spinner(self, text="Working..."):
        """
        Show a modal 'busy' indicator that doesn't freeze.
        Range 0..0 makes it an infinite spinner bar.
        """
        self.spinner = QProgressDialog(
            text,
            None,
            0, 0,
            self,
            flags=Qt.WindowTitleHint | Qt.CustomizeWindowHint
        )
        self.spinner.setWindowModality(Qt.ApplicationModal)
        self.spinner.setAutoClose(False)
        self.spinner.setCancelButton(None)
        self.spinner.setMinimumDuration(0)
        self.spinner.setRange(0, 0)
        self.spinner.show()
        QApplication.processEvents()

    def _stop_spinner(self):
        if self.spinner is not None:
            self.spinner.close()
            self.spinner = None


def main():
    app = QApplication(sys.argv)
    w = MagnetGUI()
    w.show()
    sys.exit(app.exec_())


if __name__ == "__main__":
    main()


SystemExit: 0

C:\ProgramData\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
